In [ ]:
rm service wip changes

In [ ]:
SELECT table_schema, table_name, column_name
FROM information_schema.columns
WHERE LOWER(column_name) = 'cprod_group_conformed'
ORDER BY table_schema, table_name;

In [ ]:
SELECT table_schema, table_name, column_name
FROM information_schema.columns
WHERE LOWER(table_name) LIKE 'silver_tm3%'
  AND (
       LOWER(column_name) = 'cprod_group_conformed'
       OR LOWER(column_name) LIKE '%cprod%'
       OR LOWER(column_name) LIKE '%group_conformed%'
      )
ORDER BY table_name, column_name;

In [ ]:
ffff

In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

In [ ]:
Reviewed and the implementation looks correct. For care_epi_is_test, TM3 stock_item_description is mapped to silver_rdm_care_product.cprod_name using cprod_src_sys_inst_id = 'TM3001', and the flag is set based on cprod_group_conformed = 'Test data'. This aligns with the agreed definition.

In [ ]:
For care_epi_is_test, silver_tm3_fact_appointments.stock_item_fk is used to join to silver_tm3_dim_stock_items on appt.stock_item_fk = si.stock_item_pk. From there, the stock_item_description column is taken and mapped to silver_rdm_care_product.cprod_name, with an additional join condition cprod_src_sys_inst_id = 'TM3001'. After that, the logic uses silver_rdm_care_product.cprod_group_conformed. If cprod_group_conformed = 'Test data', then care_epi_is_test is set to 1; otherwise, it is set to 0.


In [ ]:
“contr_name and contr_src_id were simple text fields, so they were mapped directly from the source dataset into the SharePoint list. But contr_src_sys_inst_id was a SharePoint lookup field, so we could not pass the text value directly. Instead, we searched the RDM - Source System Instance reference list for the matching source system instance value, took the SharePoint ID of that matching row, and passed that ID into the target lookup field. That is how SharePoint created the proper relationship and displayed the correct lookup value.”

In [ ]:
A lookup field does not store plain text. It stores the ID of the related row from the reference list. So we first found the matching source system instance row in the master list, took its SharePoint ID, and passed that ID into the target contract list lookup field.

In [ ]:
That field did not just need a value. It needed a valid link to a row in another SharePoint list, and SharePoint creates that link using the row ID.

In [ ]:
So internally, the lookup field stores something like:
“This contract row is linked to row ID 2 in the RDM - Source System Instance list.”

Then SharePoint shows the related display value on the screen:
CF001